# ML Models 04 - SHAP + Optuna

**Cel:** Zrozumiec co model przewiduje (SHAP) i poprawic parametry (Optuna).

**Mozesz uruchomic bez pelnego CV** -- wystarczy model z notebooka 02.
SHAP i Optuna dzialaja na jednym snapshotu danych.

**SHAP (SHapley Additive exPlanations):**
Jesli model przewiduje cene 5 EUR, a srednia to 3 EUR,
SHAP rozklada te 2 EUR nadwyzki na poszczegolne cechy.
Mowi ile kazda cecha 'wnosla' do predykcji.

**Optuna:** Automatycznie szuka najlepszych hiperparametrow
(num_leaves, learning_rate itp.) przez wielokrotne trenowanie modelu.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path().resolve().parents[1]))

In [ ]:
import duckdb
import sys

sys.path.insert(0, "..")

DB_PATH = "../../data/gold/cards.duckdb"
conn = duckdb.connect(DB_PATH, read_only=True)

# Latest snapshot_date for which snapshot_date + 7 days also exists (usable t+7 target).
row = conn.execute("""
    SELECT MAX(t0.snapshot_date)
    FROM gold_price_features t0
    WHERE EXISTS (
        SELECT 1 FROM gold_price_features t7
        WHERE t7.snapshot_date = CAST(t0.snapshot_date AS DATE) + INTERVAL 7 DAY
    )
""").fetchone()[0]
SNAPSHOT_DATE = str(row) if row else str(
    conn.execute("SELECT MAX(snapshot_date) FROM gold_price_features").fetchone()[0]
)
print(f"Snapshot date: {SNAPSHOT_DATE}")


## 1. SHAP Summary Plot

**Summary plot** pokazuje waznosc wszystkich cech.
Na osi X: jak bardzo dana cecha zmienia predykcje.
Kolor: wysoka (czerwona) lub niska (niebieska) wartosc cechy.

**Kluczowe pytania:**
- Czy `edhrec_saltiness` jest wysoko na liscie?
- Czy `print_count` spada gdy `saltiness` jest w modelu? (weryfikacja BA-02)
- Czy `is_reserved` ma duzy wplyw dla Tier 2?

Importuj z `src/ml/evaluation/shap_analysis.py`:

In [ ]:
from src.ml.evaluation.shap_analysis import compute_shap_values, plot_summary
from src.ml.features.lag import build_lag_features, build_target
from src.ml.features.pipeline import build_feature_pipeline, prepare_training_data
from src.ml.models.lightgbm_model import LightGBMPriceModel, LightGBMParams

lag_df = build_lag_features(conn, SNAPSHOT_DATE)
target_df = build_target(conn, SNAPSHOT_DATE)
card_df = conn.execute("SELECT * FROM gold_card_features").df()

pipeline = build_feature_pipeline()
X, y = prepare_training_data(lag_df, card_df, target_df)

params = LightGBMParams()
model = LightGBMPriceModel(params)
model.fit(X, y)

shap_values, explainer = compute_shap_values(model, X)
plot_summary(shap_values, X)

## 2. SHAP Waterfall dla 3 Kart

**Waterfall plot** pokazuje predykcje dla jednej konkretnej karty.
Wystarczy 3 karty do analizy:
1. Tania common (Basic Land) -- prosta predykcja
2. Karta z Reserved List -- `is_reserved` powinna miec wysoki SHAP
3. Tournament staple -- `edhrec_saltiness` powinien byc wazny

In [ ]:
from src.ml.evaluation.shap_analysis import plot_waterfall

# Zmien na UUIDs kart ktore chcesz analizowac
# Na razie bierzemy pierwsze 3 z X
example_indices = X.head(3).index

for i, card_idx in enumerate(example_indices):
    row = card_df[card_df["uuid"] == card_idx]
    name = row["name"].values[0] if len(row) > 0 else f"Karta {i + 1}"
    print(f"--- {name} ---")
    plot_waterfall(shap_values, X, card_idx)

## 3. Optuna -- Strojenie Hiperparametrow

**Hiperparametry do strojenia:**
- `num_leaves`: zlozonosc drzewa (31-255)
- `learning_rate`: krok uczenia (0.01-0.3)
- `min_child_samples`: min. probki w lisciu (regularyzacja)
- `feature_fraction`: ile cech na drzewo (anty-overfitting)

Kazda proba Optuna jest logowana do MLflow.
Porownaj proby: `mlflow ui` -> http://localhost:5000

n_trials=20 to dobry poczatek (~5-10 minut).

In [ ]:
from src.ml.evaluation.shap_analysis import run_optuna_tuning

best_params, study = run_optuna_tuning(
    X=X, y=y, n_trials=20, experiment_name="mtg_optuna_tuning"
)

print("Najlepsze parametry:")
for k, v in best_params.items():
    print(f"  {k}: {v}")
print(f"Najlepszy MAE: {study.best_value:.4f}")

## 4. Model z Optymalnymi Parametrami

Wytrenuj finalny model z parametrami z Optuna i zapisz go do MLflow.
MLflow run_id = identyfikator modelu. Uzywasz go w `MODEL_RUN_ID` dla API.

In [ ]:
from src.ml.training.tracking import (
    setup_experiment,
    start_run,
    log_params,
    log_metrics,
    log_model,
)
from src.ml.evaluation.metrics import mae, mape

tuned_params = LightGBMParams(**best_params)
tuned_model = LightGBMPriceModel(tuned_params)
tuned_model.fit(X, y)

setup_experiment()
with start_run(run_name="tuned_lightgbm_nb04") as run:
    log_params(vars(tuned_params))
    preds = tuned_model.predict(X)
    log_metrics({"train_mae": mae(y, preds), "train_mape": mape(y, preds)})
    run_id = log_model(tuned_model)
    print(f"Model zapisany. run_id: {run_id}")
    print("Zapisz ten run_id -- uzyjesz go w MODEL_RUN_ID w docker-compose.yml")

## Obserwacje

- Najwazniejsza cecha wg SHAP: ___
- Pozycja edhrec_saltiness: ___
- Czy print_count spada gdy saltiness w modelu (weryfikacja BA-02): ___
- Najlepsze parametry Optuna: num_leaves=___, lr=___, min_child_samples=___
- Poprawa MAE po tuningu: ___
- MLflow run_id najlepszego modelu: ___
- Wnioski: ___